# CORD-19 — Esplorazione preliminare con Dask

Setup **locale** (niente Docker): un `LocalCluster` che usa i core del Mac.
Obiettivo di questo notebook: avviare il cluster, leggere **una manciata** di file JSON
e capire lo schema dei dati prima di scrivere l'analisi vera e propria.

> Il notebook si aspetta di trovare la cartella `archive/` accanto a sé
> (cioè dentro `MAPD-Project/`). Se la sposti, aggiorna `DATA_DIR` più sotto.

## 1. Import

In [1]:
import json
from pathlib import Path

import dask
import dask.bag as db
import dask.dataframe as dd

print('dask', dask.__version__)

dask 2026.6.0


## 2. Avvio del cluster locale

`LocalCluster` crea scheduler + worker sulla tua macchina (12 core / 24 GB).
Config di partenza prudente: 4 worker × 2 thread = 8 unità, 4 GB per worker.
Regolala pure — questi sono esattamente i parametri su cui poi farai i **benchmark**.

Dopo l'avvio, apri la **dashboard** (link stampato sotto) per vedere task e memoria in tempo reale.

In [2]:
from dask.distributed import Client, LocalCluster

# chiudi un eventuale cluster precedente prima di crearne uno nuovo
try:
    client.close(); cluster.close()
except NameError:
    pass

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='4GB')
client = Client(cluster)
print('Dashboard:', client.dashboard_link)
client

Dashboard: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55315,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:55326,Total threads: 2
Dashboard: http://127.0.0.1:55328/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:55318,


## 3. Percorsi dei dati

In [3]:
from pprint import pprint

DATA_DIR = Path('archive')            # cartella del dataset (relativa al notebook)
assert DATA_DIR.exists(), f'Non trovo {DATA_DIR.resolve()} — correggi DATA_DIR'

PDF_JSON = DATA_DIR / 'document_parses' / 'pdf_json'
PMC_JSON = DATA_DIR / 'document_parses' / 'pmc_json'
METADATA = DATA_DIR / 'metadata.csv'

# NB: sono ~151k + ~112k file piccoli (~15 KB l'uno).
# Per esplorare lo schema NON leggerli tutti: prendiamo un piccolo campione.
sample_files = sorted(PDF_JSON.glob('00*.json'))[:10]
print('file campione:', len(sample_files))
pprint(sample_files)

file campione: 10
[PosixPath('archive/document_parses/pdf_json/0000028b5cc154f68b8a269f6578f21e31f62977.json'),
 PosixPath('archive/document_parses/pdf_json/00006903b396d50cc0037fed39916d57d50ee801.json'),
 PosixPath('archive/document_parses/pdf_json/0001418189999fea7f7cbe3e82703d71c85a6fe5.json'),
 PosixPath('archive/document_parses/pdf_json/00031cc15aafa30b187ed2cd3790e970e5046895.json'),
 PosixPath('archive/document_parses/pdf_json/00033d5a12240a8684cfe943954132b43434cf48.json'),
 PosixPath('archive/document_parses/pdf_json/00035ac98d8bc38fbca02a1cc957f55141af67c0.json'),
 PosixPath('archive/document_parses/pdf_json/0003793cf9e709bc2b9d0c8111186f78fb73fc04.json'),
 PosixPath('archive/document_parses/pdf_json/000379d7a7f37a2ccb978862b9f2016bd03259ea.json'),
 PosixPath('archive/document_parses/pdf_json/00039b94e6cb7609ecbddee1755314bcfeb77faa.json'),
 PosixPath('archive/document_parses/pdf_json/0003ddc51c4291d742855e9ac56076a3bea33ad7.json')]


## 4. Leggere pochi file con una Bag e ispezionare lo schema

⚠️ I JSON di CORD-19 sono **un singolo oggetto pretty-printed per file** (multi-riga),
non JSON-lines. Quindi *non* si usa `db.read_text().map(json.loads)` (che spezza per riga
e fa fallire il parsing), ma si legge **un file intero per elemento** con
`from_sequence` + `json.load`. `.take(1)` materializza un solo record, senza caricare tutto.

In [4]:
# I file CORD-19 sono UN oggetto JSON "pretty-printed" per file (multi-riga),
# NON sono JSON-lines -> si legge un intero file per elemento (non riga per riga).
def load_record(path):
    with open(path) as fh:
        return json.load(fh)

files = [str(f) for f in sample_files]
b = db.from_sequence(files).map(load_record)   # bag di dict, uno per paper

In [5]:
record = b.take(1)[0]
print('Top-level keys:', list(record.keys()))

Top-level keys: ['paper_id', 'metadata', 'abstract', 'body_text', 'bib_entries', 'ref_entries', 'back_matter']


In [6]:
record = b.take(2)[0]
print('Top-level keys:', list(record.keys()))

Top-level keys: ['paper_id', 'metadata', 'abstract', 'body_text', 'bib_entries', 'ref_entries', 'back_matter']


/Users/federicosbarbati/miniconda/envs/mapd-covid/lib/python3.11/site-packages/dask/bag/core.py:2537: UserWarning: Insufficient elements for `take`. 2 elements requested, only 1 elements available. Try passing larger `npartitions` to `take`.
  warnings.warn(


In [7]:
meta = record['metadata']
print('metadata keys :', list(meta.keys()))
print('title         :', meta['title'][:100])
print('n_authors     :', len(meta['authors']))
print('n_body_paragr :', len(record['body_text']))
print()
print('Esempio di un paragrafo di body_text:')
print(json.dumps(record['body_text'][0], indent=2)[:800])

metadata keys : ['title', 'authors']
title         : "Multi-faceted" COVID-19: Russian experience
n_authors     : 0
n_body_paragr : 13

Esempio di un paragrafo di body_text:
{
  "text": "According to current live statistics at the time of editing this letter, Russia has been the third country in the world to be affected by COVID-19 with both new cases and death rates rising. It remains in a position of advantage due to the later onset of the viral spread within the country since the worldwide disease outbreak.",
  "cite_spans": [],
  "ref_spans": [],
  "section": "Editor"
}


### Attenzione: dati sporchi

Il dataset contiene **campi vuoti e valori malformati** (es. `authors` vuoto,
`title` mancante). Nell'analisi vera dovrai filtrare/pulire in pre-processing.

In [8]:
# Quanti dei file campione hanno almeno un autore / un titolo?
recs = b.compute()
print('record letti     :', len(recs))
print('con >=1 autore   :', sum(1 for r in recs if r['metadata']['authors']))
print('con titolo       :', sum(1 for r in recs if r['metadata']['title'].strip()))

record letti     : 10
con >=1 autore   : 9
con titolo       : 10


## 5. Full-text di un paper

Per il word-count servirà il testo completo: si ottiene concatenando i `text`
dei paragrafi in `body_text` (questa sarà la base della fase *Map*).

In [9]:
full_text = ' '.join(p['text'] for p in record['body_text'])
print(len(full_text), 'caratteri')
print(full_text[:500], '...')

4258 caratteri
According to current live statistics at the time of editing this letter, Russia has been the third country in the world to be affected by COVID-19 with both new cases and death rates rising. It remains in a position of advantage due to the later onset of the viral spread within the country since the worldwide disease outbreak. The first step in "fighting" the epidemic was nationwide lock down on March 30 th , 2020. Most of the multidisciplinary hospitals have been repurposed as dedicated COVID-1 ...


## 7. Chiusura

Chiudi il cluster quando hai finito (libera memoria e porte).

In [75]:
client.close()
cluster.close()

---
### Prossimi passi (li farai tu)
- **Word count** distribuito sul full-text di tutti i paper (Bag → map/reduce).
- **Paesi/istituti** più e meno rappresentati (da `authors[].affiliation`).
- **Embeddings** dei titoli (modello pre-addestrato FastText) + **cosine similarity**.
- **Benchmark**: tempo di esecuzione al variare di `npartitions` e numero di worker.

Quando passerai al multi-nodo (Docker/CloudVeneto), cambierà solo il `Client(...)`:
la logica Dask resta identica.